In [1]:
import mysql.connector
import re
from graphviz import Digraph
import os


MYSQL_CONFIG = {
    "host": "host.docker.internal",
    "port": 3306,
    "user": "admin",
    "password": "12345678",
    "database": "spark"
}


OPERATIONS = {
    "Limit": ("LIMIT", "Limita la cantidad de filas devueltas."),
    "Offset": ("OFFSET", "Omite una cantidad de filas antes de devolver resultados."),
    "Table scan": ("TABLE SCAN", "Lee registros directamente de una tabla."),
    "Index scan": ("INDEX SCAN", "Recorre las entradas de un índice."),
    "Index range scan": ("INDEX RANGE SCAN", "Busca registros dentro de un rango de índice."),
    "Index lookup": ("INDEX LOOKUP", "Busca registros mediante un índice."),
    "Covering index lookup": ("COVERING INDEX LOOKUP", "Obtiene los datos directamente del índice."),
    "Covering index scan": ("COVERING INDEX SCAN", "Lee un índice que contiene los datos necesarios."),
    "Single-row index lookup": ("SINGLE ROW INDEX LOOKUP", "Busca un único registro mediante un índice."),
    "Nested loop inner join": ("NESTED LOOP INNER JOIN", "Busca coincidencias entre las dos entradas."),
    "Nested loop left join": ("NESTED LOOP LEFT JOIN", "Conserva las filas de la entrada izquierda."),
    "Nested loop antijoin": ("NESTED LOOP ANTIJOIN", "Busca filas de la izquierda sin coincidencias a la derecha."),
    "Hash join": ("HASH JOIN", "Combina registros utilizando una estructura hash."),
    "Filter": ("FILTER", "Descarta registros que no cumplen una condición."),
    "Sort": ("SORT", "Ordena los registros."),
    "Aggregate": ("AGGREGATE", "Calcula operaciones de agregación."),
    "Group aggregate": ("GROUP AGGREGATE", "Agrupa registros y calcula agregaciones."),
    "Materialize": ("MATERIALIZE", "Almacena temporalmente resultados."),
    "Remove duplicates": ("REMOVE DUPLICATES", "Elimina duplicados."),
    "Stream results": ("STREAM RESULTS", "Entrega los resultados."),
    "Window aggregate": ("WINDOW AGGREGATE", "Calcula agregaciones mediante funciones ventana."),
    "Window": ("WINDOW", "Ejecuta funciones ventana.")
}


AGGREGATE_FUNCTIONS = {
    "COUNT": "Cuenta registros.",
    "SUM": "Calcula la suma.",
    "AVG": "Calcula el promedio.",
    "MIN": "Obtiene el valor mínimo.",
    "MAX": "Obtiene el valor máximo.",
    "GROUP_CONCAT": "Concatena valores.",
    "STD": "Calcula la desviación estándar.",
    "STDDEV": "Calcula la desviación estándar.",
    "STDDEV_POP": "Calcula la desviación estándar poblacional.",
    "STDDEV_SAMP": "Calcula la desviación estándar muestral.",
    "VAR_POP": "Calcula la varianza poblacional.",
    "VAR_SAMP": "Calcula la varianza muestral.",
    "VARIANCE": "Calcula la varianza.",
    "BIT_AND": "Realiza una operación AND.",
    "BIT_OR": "Realiza una operación OR.",
    "BIT_XOR": "Realiza una operación XOR.",
    "JSON_ARRAYAGG": "Agrupa valores en un arreglo JSON.",
    "JSON_OBJECTAGG": "Agrupa valores en un objeto JSON."
}


WINDOW_FUNCTIONS = {
    "ROW_NUMBER": "Asigna un número consecutivo a cada fila.",
    "RANK": "Asigna posiciones con posibles saltos.",
    "DENSE_RANK": "Asigna posiciones sin saltos.",
    "NTILE": "Distribuye las filas en grupos.",
    "LAG": "Obtiene el valor de una fila anterior.",
    "LEAD": "Obtiene el valor de una fila posterior.",
    "FIRST_VALUE": "Obtiene el primer valor de la ventana.",
    "LAST_VALUE": "Obtiene el último valor del marco.",
    "NTH_VALUE": "Obtiene un valor de una posición específica.",
    "CUME_DIST": "Calcula la distribución acumulada.",
    "PERCENT_RANK": "Calcula el rango porcentual."
}


def get_operation(name):
    for key in sorted(OPERATIONS, key=len, reverse=True):
        if name.lower().startswith(key.lower()):
            return OPERATIONS[key]
    return ("OPERACIÓN MYSQL", "Operación del optimizador.")


def split_sql_expressions(expression):
    if not expression:
        return []

    results = []
    current = []
    depth = 0
    quote = None

    for char in expression:
        if quote:
            current.append(char)
            if char == quote:
                quote = None
            continue

        if char in ("'", '"', "`"):
            quote = char
            current.append(char)
        elif char == "(":
            depth += 1
            current.append(char)
        elif char == ")":
            depth -= 1
            current.append(char)
        elif char == "," and depth == 0:
            results.append("".join(current).strip())
            current = []
        else:
            current.append(char)

    if current:
        results.append("".join(current).strip())

    return results


def extract_parentheses(text, start):
    depth = 0
    quote = None

    for i in range(start, len(text)):
        char = text[i]

        if quote:
            if char == quote:
                quote = None
            continue

        if char in ("'", '"', "`"):
            quote = char
        elif char == "(":
            depth += 1
        elif char == ")":
            depth -= 1

            if depth == 0:
                return text[start + 1:i], i + 1

    return None, start


def analyze_functions(sql):
    aggregates = []
    windows = []

    functions = set(AGGREGATE_FUNCTIONS) | set(WINDOW_FUNCTIONS)

    pattern = re.compile(
        r"\b(" + "|".join(
            sorted(functions, key=len, reverse=True)
        ) + r")\s*\(",
        re.I
    )

    for match in pattern.finditer(sql):
        function = match.group(1).upper()

        arguments, end = extract_parentheses(
            sql,
            match.end() - 1
        )

        if arguments is None:
            continue

        remaining = sql[end:].lstrip()
        over_match = re.match(r"OVER\s*\(", remaining, re.I)

        if function in AGGREGATE_FUNCTIONS:
            aggregates.append({
                "function": function,
                "arguments": arguments.strip(),
                "window": bool(over_match)
            })

        if function in WINDOW_FUNCTIONS and over_match:
            over_start = end + len(sql[end:]) - len(remaining) + over_match.end() - 1

            over, _ = extract_parentheses(sql, over_start)

            if over is None:
                continue

            partition = re.search(
                r"\bPARTITION\s+BY\s+(.*?)(?=\bORDER\s+BY\b|\bROWS\b|\bRANGE\b|$)",
                over,
                re.I
            )

            ordering = re.search(
                r"\bORDER\s+BY\s+(.*?)(?=\bROWS\b|\bRANGE\b|$)",
                over,
                re.I
            )

            frame = re.search(
                r"\b(ROWS|RANGE)\s+(.+)$",
                over,
                re.I
            )

            windows.append({
                "function": function,
                "arguments": arguments.strip(),
                "partition_by": partition.group(1).strip() if partition else None,
                "order_by": ordering.group(1).strip() if ordering else None,
                "frame": frame.group(0).strip() if frame else None,
                "over": over.strip()
            })

    return aggregates, windows


def analyze_pagination(sql):
    match = re.search(
        r"\bLIMIT\s+(\d+)\s*,\s*(\d+)",
        sql,
        re.I
    )

    if match:
        return {
            "limit": int(match.group(2)),
            "offset": int(match.group(1))
        }

    match = re.search(
        r"\bLIMIT\s+(\d+)\s+OFFSET\s+(\d+)",
        sql,
        re.I
    )

    if match:
        return {
            "limit": int(match.group(1)),
            "offset": int(match.group(2))
        }

    limit_match = re.search(r"\bLIMIT\s+(\d+)", sql, re.I)
    offset_match = re.search(r"\bOFFSET\s+(\d+)", sql, re.I)

    return {
        "limit": int(limit_match.group(1)) if limit_match else None,
        "offset": int(offset_match.group(1)) if offset_match else None
    }


def analyze_sql(sql):
    clean = re.sub(r"\s+", " ", sql)

    tables = []
    joins = []

    reserved = {
        "WHERE", "LEFT", "RIGHT", "INNER",
        "OUTER", "CROSS", "JOIN", "ON",
        "ORDER", "GROUP", "LIMIT", "HAVING",
        "OFFSET", "WINDOW", "UNION"
    }

    from_match = re.search(
        r"\bFROM\s+([`\w.]+)(?:\s+(?:AS\s+)?(\w+))?",
        clean,
        re.I
    )

    if from_match:
        table = from_match.group(1).replace("`", "")
        alias = from_match.group(2)

        if alias and alias.upper() in reserved:
            alias = None

        tables.append({
            "table": table,
            "alias": alias or table.split(".")[-1]
        })

    pattern = re.compile(
        r"\b(INNER|LEFT|RIGHT|CROSS)?\s*"
        r"(?:OUTER\s+)?JOIN\s+([`\w.]+)"
        r"(?:\s+(?:AS\s+)?(\w+))?\s+ON\s+(.*?)"
        r"(?=\s+(?:(?:INNER|LEFT|RIGHT|CROSS)\s+)?"
        r"(?:OUTER\s+)?JOIN\b|\s+WHERE\b|"
        r"\s+GROUP BY\b|\s+HAVING\b|"
        r"\s+ORDER BY\b|\s+LIMIT\b|"
        r"\s+OFFSET\b|$)",
        re.I
    )

    for match in pattern.finditer(clean):
        join_type = (match.group(1) or "INNER").upper()
        table = match.group(2).replace("`", "")
        alias = match.group(3)

        if alias and alias.upper() in reserved:
            alias = None

        alias = alias or table.split(".")[-1]

        tables.append({
            "table": table,
            "alias": alias
        })

        joins.append({
            "type": join_type,
            "table": table,
            "alias": alias,
            "condition": match.group(4).strip()
        })

    where = re.search(
        r"\bWHERE\s+(.*?)(?=\s+GROUP BY\b|\s+HAVING\b|\s+WINDOW\b|\s+ORDER BY\b|\s+LIMIT\b|\s+OFFSET\b|$)",
        clean,
        re.I
    )

    group_by = re.search(
        r"\bGROUP BY\s+(.*?)(?=\s+HAVING\b|\s+WINDOW\b|\s+ORDER BY\b|\s+LIMIT\b|\s+OFFSET\b|$)",
        clean,
        re.I
    )

    having = re.search(
        r"\bHAVING\s+(.*?)(?=\s+WINDOW\b|\s+ORDER BY\b|\s+LIMIT\b|\s+OFFSET\b|$)",
        clean,
        re.I
    )

    order_by = re.search(
        r"\bORDER BY\s+(.*?)(?=\s+LIMIT\b|\s+OFFSET\b|$)",
        clean,
        re.I
    )

    window_clause = re.search(
        r"\bWINDOW\s+(.*?)(?=\s+ORDER BY\b|\s+LIMIT\b|\s+OFFSET\b|$)",
        clean,
        re.I
    )

    select = re.search(
        r"\bSELECT\s+(.*?)\s+FROM\b",
        clean,
        re.I
    )

    aggregates, windows = analyze_functions(clean)
    pagination = analyze_pagination(clean)

    return {
        "tables": tables,
        "joins": joins,
        "where": where.group(1) if where else None,
        "select": split_sql_expressions(select.group(1)) if select else [],
        "group_by": group_by.group(1) if group_by else None,
        "having": having.group(1) if having else None,
        "order_by": order_by.group(1) if order_by else None,
        "window_clause": window_clause.group(1) if window_clause else None,
        "limit": pagination["limit"],
        "offset": pagination["offset"],
        "aggregates": aggregates,
        "windows": windows
    }


def parse_line(line):
    name = re.sub(r"^\s*->\s*", "", line).strip()

    patterns = {
        "cost": r"cost=([\d.]+)",
        "estimated": r"cost=[\d.]+\s+rows=(\d+)",
        "start": r"actual time=([\d.]+)",
        "end": r"actual time=[\d.]+\.\.([\d.]+)",
        "actual": r"actual time=[\d.]+\.\.[\d.]+\s+rows=(\d+)",
        "loops": r"loops=(\d+)"
    }

    metrics = {}

    for key, pattern in patterns.items():
        match = re.search(pattern, line)
        metrics[key] = match.group(1) if match else None

    name = re.sub(r"\s+\(cost=.*$", "", name)

    return name, metrics


def build_tree(plan_text):
    root = None
    stack = []

    for line in plan_text.splitlines():
        if not line.strip():
            continue

        indent = len(line) - len(line.lstrip())
        depth = indent // 4

        name, metrics = parse_line(line)

        node = {
            "name": name,
            "depth": depth,
            "metrics": metrics,
            "children": []
        }

        while stack and stack[-1]["depth"] >= depth:
            stack.pop()

        if stack:
            stack[-1]["children"].append(node)
        else:
            root = node

        stack.append(node)

    return root


def get_execution_sequence(node, sequence=None):
    if sequence is None:
        sequence = []

    for child in node["children"]:
        get_execution_sequence(child, sequence)

    sequence.append(node)

    return sequence


def find_table(name, tables):
    for item in tables:
        alias = re.escape(item["alias"])

        if re.search(
            r"\b" + alias + r"\b",
            name.split(" using ")[0],
            re.I
        ):
            return item["table"]

    return None


def collect_tables(node, tables):
    found = []

    table = find_table(node["name"], tables)

    if table:
        found.append(table)

    for child in node["children"]:
        found.extend(collect_tables(child, tables))

    return list(dict.fromkeys(found))


def describe_join(node, sql_info):
    name = node["name"].lower()

    if "join" not in name:
        return

    print("\n      RELACIÓN ENTRE ENTRADAS")

    if len(node["children"]) >= 2:
        left = collect_tables(node["children"][0], sql_info["tables"])
        right = collect_tables(node["children"][1], sql_info["tables"])

        print("      Entrada izquierda:", ", ".join(left) or "Resultado intermedio")
        print("      Entrada derecha:", ", ".join(right) or "Resultado intermedio")
    else:
        left = []
        right = []

    matching = None

    for join in sql_info["joins"]:
        if join["table"] in right:
            matching = join
            break

    if matching:
        print(f"      JOIN SQL: {matching['type']} JOIN")
        print(f"      Tabla derecha: {matching['table']}")
        print(f"      Alias: {matching['alias']}")
        print(f"      Condición ON: {matching['condition']}")
    else:
        print("      JOIN SQL: No identificado inequívocamente.")

    if "antijoin" in name:
        print("      Operación física: ANTIJOIN")
        print("      Conserva filas de la entrada izquierda sin coincidencias en la derecha.")

    elif "nested loop" in name:
        print("      Operación física: NESTED LOOP")
        print("      Busca coincidencias mediante búsquedas repetidas en la entrada interna.")

    elif "hash join" in name:
        print("      Operación física: HASH JOIN")
        print("      Utiliza una estructura hash para relacionar ambas entradas.")


# ============================================================
# GENERACIÓN DEL GRAFO CON GRAPHVIZ
# ============================================================


def generate_graphviz(root):
    graph = Digraph("Execution Plan", format="png")

    # Árbol de abajo hacia arriba
    graph.attr(
        rankdir="BT",
        ordering="out"
    )

    graph.attr(
        "graph",
        bgcolor="white",
        nodesep="0.5",
        ranksep="0.7",
        splines="ortho"
    )

    graph.attr(
        "node",
        shape="box",
        style="filled,rounded",
        fontname="Arial",
        fontsize="10",
        margin="0.15",
        color="#333333",
        penwidth="1.5"
    )

    graph.attr(
        "edge",
        color="#555555",
        arrowsize="0.8"
    )

    # Color diferente para cada operación
    COLORS = {
        "LIMIT": "#FFCC80",
        "OFFSET": "#FFE082",
        "TABLE SCAN": "#90CAF9",
        "INDEX SCAN": "#80DEEA",
        "INDEX RANGE SCAN": "#80CBC4",
        "INDEX LOOKUP": "#A5D6A7",
        "COVERING INDEX LOOKUP": "#C5E1A5",
        "COVERING INDEX SCAN": "#AED581",
        "SINGLE ROW INDEX LOOKUP": "#B2DFDB",
        "NESTED LOOP INNER JOIN": "#CE93D8",
        "NESTED LOOP LEFT JOIN": "#F48FB1",
        "NESTED LOOP ANTIJOIN": "#EF9A9A",
        "HASH JOIN": "#B39DDB",
        "FILTER": "#FFF59D",
        "SORT": "#FFAB91",
        "AGGREGATE": "#80CBC4",
        "GROUP AGGREGATE": "#4DB6AC",
        "MATERIALIZE": "#B0BEC5",
        "REMOVE DUPLICATES": "#F8BBD0",
        "STREAM RESULTS": "#C5CAE9",
        "WINDOW AGGREGATE": "#9FA8DA",
        "WINDOW": "#7986CB"
    }

    counter = [0]

    def add_nodes(node, parent_id=None):

        counter[0] += 1
        node_id = f"node_{counter[0]}"

        operation, description = get_operation(node["name"])
        metrics = node["metrics"]

        # Color según el tipo de operación
        color = COLORS.get(
            operation,
            "#E0E0E0"
        )

        label = (
            f"{operation}\n"
            f"{description}\n"
            f"Operación: {node['name']}"
        )

        if metrics["cost"]:
            label += f"\nCoste: {metrics['cost']}"

        if metrics["estimated"]:
            label += f"\nFilas estimadas: {metrics['estimated']}"

        if metrics["actual"]:
            label += f"\nFilas reales: {metrics['actual']}"

        if metrics["start"] and metrics["end"]:
            label += (
                f"\nTiempo: "
                f"{metrics['start']}..{metrics['end']} ms"
            )

        if metrics["loops"]:
            label += f"\nLoops: {metrics['loops']}"

        graph.node(
            node_id,
            label=label,
            fillcolor=color
        )

        if parent_id is not None:
            graph.edge(
                parent_id,
                node_id
            )

        # Mantener el orden original de los hijos
        for child in node["children"]:
            add_nodes(child, node_id)

    add_nodes(root)

    output_path = graph.render(
        filename="execution_plan",
        cleanup=True,
        view=True
    )

    print("\nIMAGEN DEL PLAN DE EJECUCIÓN GENERADA:")
    print(os.path.abspath(output_path))

    return output_path



# ============================================================
# ANÁLISIS DE CLÁUSULAS Y FUNCIONES SQL
# ============================================================

def explain_sql_operations(sql_info):

    print("\n" + "=" * 100)
    print("ANÁLISIS DE CLÁUSULAS Y FUNCIONES SQL")
    print("=" * 100)

    print("\nSELECT")

    for column in sql_info["select"]:
        print("   ", column)

    print("\nWHERE")
    print("   ", sql_info["where"] or "Sin WHERE")

    print("\nGROUP BY")

    if sql_info["group_by"]:
        for column in split_sql_expressions(sql_info["group_by"]):
            print("   ", column)
    else:
        print("   Sin GROUP BY")

    print("\nHAVING")
    print("   ", sql_info["having"] or "Sin HAVING")

    print("\nFUNCIONES DE AGREGACIÓN")

    if sql_info["aggregates"]:
        for item in sql_info["aggregates"]:
            print(
                f"   {item['function']}({item['arguments']})"
            )

            print(
                "   ",
                AGGREGATE_FUNCTIONS[item["function"]]
            )

            if item["window"]:
                print("    Aplicada como función ventana.")
    else:
        print("   No se detectaron agregaciones.")

    print("\nFUNCIONES VENTANA")

    if sql_info["windows"]:
        for item in sql_info["windows"]:

            print("\n   Función:", item["function"])
            print("   Argumentos:", item["arguments"])

            print(
                "   Descripción:",
                WINDOW_FUNCTIONS[item["function"]]
            )

            print(
                "   PARTITION BY:",
                item["partition_by"] or "No definido"
            )

            print(
                "   ORDER BY:",
                item["order_by"] or "No definido"
            )

            print(
                "   FRAME:",
                item["frame"] or "Marco por defecto"
            )

            print("   OVER:", item["over"])

    else:
        print("   No se detectaron funciones ventana.")

    print("\nWINDOW")
    print("   ", sql_info["window_clause"] or "Sin WINDOW")

    print("\nORDER BY")
    print("   ", sql_info["order_by"] or "Sin ORDER BY")

    print("\nLIMIT")
    print(
        "   ",
        sql_info["limit"] if sql_info["limit"] is not None else "Sin LIMIT"
    )

    print("\nOFFSET")
    print(
        "   ",
        sql_info["offset"] if sql_info["offset"] is not None else 0
    )


def explain_tree(explain_result, sql_query):
    sql_info = analyze_sql(sql_query)

    for row in explain_result:
        plan_text = row[0] if isinstance(row, (tuple, list)) else str(row)

        root = build_tree(plan_text)

        if not root:
            continue

        # GENERACIÓN DE LA IMAGEN CON GRAPHVIZ
        generate_graphviz(root)

        print("\n" + "=" * 100)
        print("ANÁLISIS DEL PLAN DE EJECUCIÓN MYSQL")
        print("=" * 100)

        print("\n1. TABLAS INVOLUCRADAS")

        for table in sql_info["tables"]:
            print(f"   {table['table']} (alias: {table['alias']})")

        print("\n2. JOINS DECLARADOS")

        for i, join in enumerate(sql_info["joins"], 1):
            print(f"\n   JOIN #{i}")
            print(f"   Tipo: {join['type']} JOIN")
            print(f"   Tabla: {join['table']}")
            print(f"   Condición: {join['condition']}")

        print("\n3. CONDICIÓN WHERE")
        print("  ", sql_info["where"] or "Sin WHERE")

        sequence = get_execution_sequence(root)

        print("\n" + "=" * 100)
        print("4. OPERACIONES EN ORDEN DE EJECUCIÓN")
        print("=" * 100)

        for i, node in enumerate(sequence, 1):
            operation, description = get_operation(node["name"])
            metrics = node["metrics"]

            print(f"\n{'─' * 90}")
            print(f"PASO {i}: {operation}")
            print(f"{'─' * 90}")

            print("Operación física:", node["name"])
            print("Descripción:", description)

            table = find_table(node["name"], sql_info["tables"])

            if table:
                print("Tabla consultada:", table)

            index = re.search(r"\busing\s+([^\s(]+)", node["name"], re.I)

            if index:
                print("Índice utilizado:", index.group(1))

            if "join" in node["name"].lower():
                describe_join(node, sql_info)

            if metrics["cost"]:
                print("\nMÉTRICAS")
                print("Coste estimado:", metrics["cost"], "unidades internas")
                print("Filas estimadas:", metrics["estimated"])

            if metrics["actual"]:
                actual = int(metrics["actual"])
                loops = int(metrics["loops"] or 1)

                print("Filas reales por ejecución:", actual)
                print("Tiempo inicial:", metrics["start"], "ms")
                print("Tiempo final:", metrics["end"], "ms")
                print("Loops:", loops)
                print("Filas acumuladas aproximadas:", actual * loops)

        print("\n" + "=" * 100)
        print("5. ÁRBOL JERÁRQUICO")
        print("=" * 100)

        def render(node, prefix="", last=True, root_node=False):
            operation, _ = get_operation(node["name"])

            connector = "" if root_node else ("└── " if last else "├── ")

            print(prefix + connector + operation)

            new_prefix = prefix + (
                "    " if last or root_node else "│   "
            )

            for i, child in enumerate(node["children"]):
                render(
                    child,
                    new_prefix,
                    i == len(node["children"]) - 1
                )

        render(root, root_node=True)

        # ANÁLISIS DE GROUP BY, HAVING,
        # AGREGACIONES, FUNCIONES VENTANA Y OFFSET

        explain_sql_operations(sql_info)

        print("\n" + "=" * 100)


def execute_query(
    query,
    params=None,
    show_results=False,
    explain_analyse=False,
    show_query=False
):
    connection = None
    cursor = None

    try:
        connection = mysql.connector.connect(**MYSQL_CONFIG)
        cursor = connection.cursor()

        query = re.sub(
            r"^\s*EXPLAIN\s+ANALYZE\s+",
            "",
            query,
            flags=re.I
        ).strip().rstrip(";")

        final_query = (
            "EXPLAIN ANALYZE " + query
            if explain_analyse
            else query
        )

        if show_query:
            print("\nCONSULTA EJECUTADA:\n")
            print(final_query)

        cursor.execute(final_query, params or ())

        if cursor.with_rows:
            results = cursor.fetchall()

            if explain_analyse:
                explain_tree(results, sql_query=query)

            elif show_results:
                for row in results:
                    print(row)

            return results

        connection.commit()
        return cursor.rowcount

    except mysql.connector.Error as e:
        if connection and connection.is_connected():
            connection.rollback()

        print("ERROR MYSQL:", e)
        return None

    except Exception as e:
        print("ERROR:", e)
        return None

    finally:
        if cursor:
            cursor.close()

        if connection and connection.is_connected():
            connection.close()


# ============================================================
# EJEMPLO DE USO
# ============================================================

query = """

SELECT
    c.category_id,
    c.name AS category_name,

    COUNT(p.product_id) AS total_products,
    SUM(p.stock) AS total_stock,
    ROUND(AVG(p.price), 2) AS average_price,
    MIN(p.price) AS minimum_price,
    MAX(p.price) AS maximum_price,

    ROW_NUMBER() OVER (
        ORDER BY COUNT(p.product_id) DESC
    ) AS numero_fila,

    RANK() OVER (
        ORDER BY AVG(p.price) DESC
    ) AS ranking_precio,

    DENSE_RANK() OVER (
        ORDER BY SUM(p.stock) DESC
    ) AS ranking_stock,

    LAG(AVG(p.price)) OVER (
        ORDER BY c.category_id
    ) AS precio_categoria_anterior,

    LEAD(AVG(p.price)) OVER (
        ORDER BY c.category_id
    ) AS precio_categoria_siguiente,

    SUM(COUNT(p.product_id)) OVER () AS total_general

FROM categories c

INNER JOIN products p
    ON c.category_id = p.category_id

GROUP BY
    c.category_id,
    c.name

HAVING
    COUNT(p.product_id) >= 1
    AND SUM(p.stock) > 0
    AND AVG(p.price) > 0

ORDER BY
    total_products DESC,
    average_price DESC

LIMIT 100
OFFSET 1;
"""

result = execute_query(
    query=query,
    explain_analyse=True,
    show_query=True
)


CONSULTA EJECUTADA:

EXPLAIN ANALYZE SELECT
    c.category_id,
    c.name AS category_name,

    COUNT(p.product_id) AS total_products,
    SUM(p.stock) AS total_stock,
    ROUND(AVG(p.price), 2) AS average_price,
    MIN(p.price) AS minimum_price,
    MAX(p.price) AS maximum_price,

    ROW_NUMBER() OVER (
        ORDER BY COUNT(p.product_id) DESC
    ) AS numero_fila,

    RANK() OVER (
        ORDER BY AVG(p.price) DESC
    ) AS ranking_precio,

    DENSE_RANK() OVER (
        ORDER BY SUM(p.stock) DESC
    ) AS ranking_stock,

    LAG(AVG(p.price)) OVER (
        ORDER BY c.category_id
    ) AS precio_categoria_anterior,

    LEAD(AVG(p.price)) OVER (
        ORDER BY c.category_id
    ) AS precio_categoria_siguiente,

    SUM(COUNT(p.product_id)) OVER () AS total_general

FROM categories c

INNER JOIN products p
    ON c.category_id = p.category_id

GROUP BY
    c.category_id,
    c.name

HAVING
    COUNT(p.product_id) >= 1
    AND SUM(p.stock) > 0
    AND AVG(p.price) > 0

ORDER

In [2]:
import mysql.connector
import re
import os
from graphviz import Digraph


MYSQL_CONFIG = {
    "host": "host.docker.internal",
    "port": 3306,
    "user": "admin",
    "password": "12345678",
    "database": "spark"
}


OPERATIONS = {
    "Limit": ("LIMIT", "Limita la cantidad de filas devueltas."),
    "Offset": ("OFFSET", "Omite una cantidad determinada de filas."),
    "Table scan": ("TABLE SCAN", "Lee registros directamente de una tabla."),
    "Index scan": ("INDEX SCAN", "Recorre las entradas de un índice."),
    "Index range scan": ("INDEX RANGE SCAN", "Busca registros dentro de un rango de índice."),
    "Index lookup": ("INDEX LOOKUP", "Busca registros mediante un índice."),
    "Covering index lookup": ("COVERING INDEX LOOKUP", "Obtiene datos directamente del índice."),
    "Covering index scan": ("COVERING INDEX SCAN", "Lee un índice que contiene los datos necesarios."),
    "Single-row index lookup": ("SINGLE ROW INDEX LOOKUP", "Busca un único registro mediante un índice."),
    "Nested loop inner join": ("NESTED LOOP INNER JOIN", "Relaciona filas coincidentes de ambas entradas."),
    "Nested loop left join": ("NESTED LOOP LEFT JOIN", "Conserva las filas de la entrada izquierda."),
    "Nested loop antijoin": ("NESTED LOOP ANTIJOIN", "Busca filas sin coincidencias en la entrada derecha."),
    "Hash join": ("HASH JOIN", "Combina registros utilizando una estructura hash."),
    "Filter": ("FILTER", "Descarta registros que no cumplen una condición."),
    "Sort": ("SORT", "Ordena los registros."),
    "Aggregate": ("AGGREGATE", "Calcula operaciones de agregación."),
    "Group aggregate": ("GROUP AGGREGATE", "Agrupa registros y calcula agregaciones."),
    "Materialize": ("MATERIALIZE", "Almacena temporalmente resultados."),
    "Remove duplicates": ("REMOVE DUPLICATES", "Elimina duplicados."),
    "Stream results": ("STREAM RESULTS", "Entrega los resultados."),
    "Window aggregate": ("WINDOW AGGREGATE", "Calcula funciones ventana."),
    "Window": ("WINDOW", "Procesa funciones ventana.")
}


AGGREGATE_FUNCTIONS = {
    "COUNT": "Cuenta registros o valores no nulos.",
    "SUM": "Suma los valores de una columna.",
    "AVG": "Calcula el promedio de una columna.",
    "MIN": "Obtiene el valor mínimo.",
    "MAX": "Obtiene el valor máximo.",
    "GROUP_CONCAT": "Concatena valores de un grupo.",
    "STD": "Calcula la desviación estándar.",
    "STDDEV": "Calcula la desviación estándar.",
    "VARIANCE": "Calcula la varianza.",
    "JSON_ARRAYAGG": "Agrupa valores en un arreglo JSON.",
    "JSON_OBJECTAGG": "Agrupa pares clave-valor en un objeto JSON."
}


WINDOW_FUNCTIONS = {
    "ROW_NUMBER": "Asigna un número consecutivo a cada fila.",
    "RANK": "Clasifica las filas, dejando saltos en los empates.",
    "DENSE_RANK": "Clasifica las filas sin saltos.",
    "NTILE": "Divide las filas en grupos.",
    "LAG": "Obtiene un valor de una fila anterior.",
    "LEAD": "Obtiene un valor de una fila posterior.",
    "FIRST_VALUE": "Obtiene el primer valor de la ventana.",
    "LAST_VALUE": "Obtiene el último valor de la ventana.",
    "NTH_VALUE": "Obtiene un valor de una posición determinada.",
    "CUME_DIST": "Calcula la distribución acumulada.",
    "PERCENT_RANK": "Calcula la clasificación porcentual."
}


def get_operation(name):

    for key in sorted(OPERATIONS, key=len, reverse=True):

        if name.lower().startswith(key.lower()):
            return OPERATIONS[key]

    return ("OPERACIÓN MYSQL", "Operación del optimizador.")


def split_sql_expressions(expression):

    if not expression:
        return []

    result = []
    current = []
    depth = 0
    quote = None

    for char in expression:

        if char in ("'", '"', '`'):

            if quote == char:
                quote = None
            elif quote is None:
                quote = char

        elif quote is None:

            if char == "(":
                depth += 1

            elif char == ")":
                depth -= 1

            elif char == "," and depth == 0:

                result.append("".join(current).strip())
                current = []
                continue

        current.append(char)

    if current:
        result.append("".join(current).strip())

    return result


def extract_sql_clause(sql, clause, next_clauses):

    pattern = (
        r"\b" + clause + r"\b\s+"
        r"(.*?)"
        r"(?=\s+(?:" + "|".join(next_clauses) + r")\b|$)"
    )

    match = re.search(pattern, sql, re.I)

    return match.group(1).strip() if match else None


def analyze_sql(sql):

    clean = re.sub(r"\s+", " ", sql).strip()

    tables = []
    joins = []

    reserved = {
        "WHERE", "LEFT", "RIGHT", "INNER",
        "OUTER", "CROSS", "JOIN", "ON",
        "ORDER", "GROUP", "LIMIT", "HAVING",
        "OFFSET", "WINDOW", "SET", "UNION"
    }

    # TABLA PRINCIPAL

    from_match = re.search(
        r"\bFROM\s+([`\w.]+)(?:\s+(?:AS\s+)?(\w+))?",
        clean,
        re.I
    )

    if from_match:

        table = from_match.group(1).replace("`", "")
        alias = from_match.group(2)

        if alias and alias.upper() in reserved:
            alias = None

        tables.append({
            "table": table,
            "alias": alias or table.split(".")[-1]
        })

    # JOINS

    join_pattern = re.compile(
        r"\b(INNER|LEFT|RIGHT|CROSS)?\s*"
        r"(?:OUTER\s+)?JOIN\s+([`\w.]+)"
        r"(?:\s+(?:AS\s+)?(\w+))?\s+ON\s+(.*?)"
        r"(?=\s+(?:(?:INNER|LEFT|RIGHT|CROSS)\s+)?"
        r"(?:OUTER\s+)?JOIN\b|"
        r"\s+WHERE\b|\s+GROUP BY\b|"
        r"\s+HAVING\b|\s+WINDOW\b|"
        r"\s+ORDER BY\b|\s+LIMIT\b|"
        r"\s+OFFSET\b|$)",
        re.I
    )

    for match in join_pattern.finditer(clean):

        join_type = (match.group(1) or "INNER").upper()

        table = match.group(2).replace("`", "")

        alias = match.group(3)

        if alias and alias.upper() in reserved:
            alias = None

        alias = alias or table.split(".")[-1]

        tables.append({
            "table": table,
            "alias": alias
        })

        joins.append({
            "type": join_type,
            "table": table,
            "alias": alias,
            "condition": match.group(4).strip()
        })

    # SELECT

    select_match = re.search(
        r"\bSELECT\s+(.*?)\s+FROM\b",
        clean,
        re.I
    )

    select = (
        split_sql_expressions(select_match.group(1))
        if select_match else []
    )

    # CLÁUSULAS SQL

    where = extract_sql_clause(
        clean, "WHERE",
        ["GROUP BY", "HAVING", "WINDOW", "ORDER BY", "LIMIT", "OFFSET"]
    )

    group_by = extract_sql_clause(
        clean, "GROUP BY",
        ["HAVING", "WINDOW", "ORDER BY", "LIMIT", "OFFSET"]
    )

    having = extract_sql_clause(
        clean, "HAVING",
        ["WINDOW", "ORDER BY", "LIMIT", "OFFSET"]
    )

    order_by = extract_sql_clause(
        clean, "ORDER BY",
        ["LIMIT", "OFFSET"]
    )

    limit_match = re.search(
        r"\bLIMIT\s+(\d+)",
        clean,
        re.I
    )

    offset_match = re.search(
        r"\bOFFSET\s+(\d+)",
        clean,
        re.I
    )

    # FUNCIONES DE AGREGACIÓN

    aggregates = []

    aggregate_pattern = re.compile(
        r"\b(COUNT|SUM|AVG|MIN|MAX|GROUP_CONCAT|"
        r"STD|STDDEV|VARIANCE|JSON_ARRAYAGG|JSON_OBJECTAGG)"
        r"\s*\((.*?)\)",
        re.I
    )

    for match in aggregate_pattern.finditer(clean):

        function = match.group(1).upper()
        arguments = match.group(2)

        after = clean[match.end():]

        is_window = bool(
            re.match(r"\s*OVER\s*\(", after, re.I)
        )

        aggregates.append({
            "function": function,
            "arguments": arguments,
            "window": is_window
        })

    # FUNCIONES VENTANA

    windows = []

    window_pattern = re.compile(
        r"\b(ROW_NUMBER|RANK|DENSE_RANK|NTILE|"
        r"LAG|LEAD|FIRST_VALUE|LAST_VALUE|NTH_VALUE|"
        r"CUME_DIST|PERCENT_RANK|COUNT|SUM|AVG|MIN|MAX)"
        r"\s*\((.*?)\)\s*OVER\s*\((.*?)\)",
        re.I
    )

    for match in window_pattern.finditer(clean):

        function = match.group(1).upper()
        arguments = match.group(2)
        over = match.group(3)

        partition_match = re.search(
            r"\bPARTITION BY\s+(.*?)(?=\s+ORDER BY\b|\s+ROWS\b|\s+RANGE\b|$)",
            over,
            re.I
        )

        order_match = re.search(
            r"\bORDER BY\s+(.*?)(?=\s+ROWS\b|\s+RANGE\b|$)",
            over,
            re.I
        )

        frame_match = re.search(
            r"\b(ROWS|RANGE|GROUPS)\s+.*",
            over,
            re.I
        )

        windows.append({
            "function": function,
            "arguments": arguments,
            "over": over,
            "partition_by": partition_match.group(1).strip() if partition_match else None,
            "order_by": order_match.group(1).strip() if order_match else None,
            "frame": frame_match.group(0).strip() if frame_match else None
        })

    return {
        "tables": tables,
        "joins": joins,
        "select": select,
        "where": where,
        "group_by": group_by,
        "having": having,
        "order_by": order_by,
        "limit": limit_match.group(1) if limit_match else None,
        "offset": offset_match.group(1) if offset_match else None,
        "aggregates": aggregates,
        "windows": windows
    }


def resolve_real_table(alias, tables):

    alias = alias.replace("`", "")

    for item in tables:

        if item["alias"].lower() == alias.lower():
            return item["table"]

        if item["table"].split(".")[-1].lower() == alias.lower():
            return item["table"]

    return alias


def resolve_column_reference(expression, tables):

    expression = expression.strip().replace("`", "")

    match = re.match(
        r"^(\w+)\.(\w+)$",
        expression
    )

    if not match:
        return expression

    real_table = resolve_real_table(
        match.group(1),
        tables
    )

    return f"{real_table}.{match.group(2)}"


def extract_columns(expression, tables):

    references = re.findall(
        r"\b(\w+)\.(\w+)\b",
        expression
    )

    columns = []

    for alias, column in references:

        resolved = resolve_column_reference(
            f"{alias}.{column}",
            tables
        )

        if resolved not in columns:
            columns.append(resolved)

    return columns


def parse_line(line):

    name = re.sub(
        r"^\s*->\s*",
        "",
        line
    ).strip()

    patterns = {
        "cost": r"cost=([\d.]+)",
        "estimated": r"rows=(\d+)",
        "start": r"actual time=([\d.]+)",
        "end": r"actual time=[\d.]+\.\.([\d.]+)",
        "actual": r"actual time=[\d.]+\.\.[\d.]+\s+rows=(\d+)",
        "loops": r"loops=(\d+)"
    }

    metrics = {}

    for key, pattern in patterns.items():

        match = re.search(pattern, line)

        metrics[key] = match.group(1) if match else None

    name = re.sub(
        r"\s+\(cost=.*$",
        "",
        name
    )

    return name, metrics

# ============================================================
# FUNCIONES AUXILIARES PARA IDENTIFICAR TABLAS Y COLUMNAS
# ============================================================

def resolve_table_name(reference, tables):
    reference = reference.replace("`", "").strip()

    if "." not in reference:
        return reference

    qualifier, column = reference.rsplit(".", 1)

    for item in tables:
        if qualifier.lower() in (
            item["alias"].lower(),
            item["table"].lower(),
            item["table"].split(".")[-1].lower()
        ):
            return item["table"] + "." + column

    return reference


def resolve_column_reference(reference, tables):
    return resolve_table_name(reference, tables)


def split_sql_expressions(expression):
    if not expression:
        return []

    parts = []
    current = []
    depth = 0
    quote = None

    for char in expression:
        if quote:
            current.append(char)
            if char == quote:
                quote = None
            continue

        if char in ("'", '"'):
            quote = char
            current.append(char)
        elif char == "(":
            depth += 1
            current.append(char)
        elif char == ")":
            depth -= 1
            current.append(char)
        elif char == "," and depth == 0:
            parts.append("".join(current).strip())
            current = []
        else:
            current.append(char)

    if current:
        parts.append("".join(current).strip())

    return parts


def extract_columns(expression, tables):
    if not expression:
        return []

    pattern = re.compile(
        r'(?:(?:`?[\w]+`?)\s*\.\s*)?`?[\w]+`?'
    )

    results = []

    keywords = {
        "COUNT", "SUM", "AVG", "MIN", "MAX",
        "DISTINCT", "AS", "OVER", "PARTITION",
        "BY", "ORDER", "ROWS", "RANGE",
        "BETWEEN", "PRECEDING", "FOLLOWING",
        "CURRENT", "ROW", "UNBOUNDED",
        "ASC", "DESC", "CASE", "WHEN",
        "THEN", "ELSE", "END", "AND",
        "OR", "NOT", "IS", "NULL"
    }

    for match in pattern.finditer(expression):
        reference = match.group(0).replace("`", "")

        if reference.upper() in keywords:
            continue

        end = match.end()

        if end < len(expression):
            if expression[end:].lstrip().startswith("("):
                continue

        resolved = resolve_column_reference(
            reference,
            tables
        )

        if resolved not in results:
            results.append(resolved)

    return results


# ============================================================
# FUNCIONES DE AGREGACIÓN
# ============================================================

AGGREGATE_FUNCTIONS = {
    "COUNT": "Cuenta filas o valores no nulos.",
    "SUM": "Calcula la suma de los valores.",
    "AVG": "Calcula el promedio.",
    "MIN": "Obtiene el valor mínimo.",
    "MAX": "Obtiene el valor máximo.",
    "GROUP_CONCAT": "Concatena valores de un grupo."
}


# ============================================================
# FUNCIONES VENTANA
# ============================================================

WINDOW_FUNCTIONS = {
    "ROW_NUMBER": "Asigna un número consecutivo a cada fila.",
    "RANK": "Asigna posiciones y deja saltos en los empates.",
    "DENSE_RANK": "Asigna posiciones sin saltos.",
    "NTILE": "Distribuye filas en grupos.",
    "LAG": "Obtiene un valor de una fila anterior.",
    "LEAD": "Obtiene un valor de una fila posterior.",
    "FIRST_VALUE": "Obtiene el primer valor de la ventana.",
    "LAST_VALUE": "Obtiene el último valor del marco.",
    "NTH_VALUE": "Obtiene un valor de una posición específica.",
    "CUME_DIST": "Calcula la distribución acumulada.",
    "PERCENT_RANK": "Calcula el rango porcentual."
}


def analyze_functions(sql):
    aggregates = []
    windows = []

    window_names = "|".join(WINDOW_FUNCTIONS)

    window_pattern = re.compile(
        rf'\b({window_names})\s*\((.*?)\)\s*OVER\s*\((.*?)\)',
        re.I
    )

    for match in window_pattern.finditer(sql):
        function = match.group(1).upper()
        arguments = match.group(2).strip()
        over = match.group(3).strip()

        partition = re.search(
            r'\bPARTITION\s+BY\s+(.*?)(?=\bORDER\s+BY\b|\bROWS\b|\bRANGE\b|$)',
            over,
            re.I
        )

        order = re.search(
            r'\bORDER\s+BY\s+(.*?)(?=\bROWS\b|\bRANGE\b|$)',
            over,
            re.I
        )

        frame = re.search(
            r'\b(ROWS|RANGE)\s+.*$',
            over,
            re.I
        )

        windows.append({
            "function": function,
            "arguments": arguments,
            "over": over,
            "partition_by": partition.group(1).strip() if partition else None,
            "order_by": order.group(1).strip() if order else None,
            "frame": frame.group(0).strip() if frame else None
        })

    aggregate_pattern = re.compile(
        r'\b(COUNT|SUM|AVG|MIN|MAX|GROUP_CONCAT)\s*\((.*?)\)',
        re.I
    )

    for match in aggregate_pattern.finditer(sql):
        function = match.group(1).upper()

        is_window = bool(
            re.search(
                r'\bOVER\s*\(',
                sql[match.end():match.end() + 200],
                re.I
            )
        )

        aggregates.append({
            "function": function,
            "arguments": match.group(2).strip(),
            "window": is_window
        })

    return aggregates, windows


# ============================================================
# ANALIZAR SQL COMPLETO
# ============================================================

def analyze_sql(sql):
    clean = re.sub(r"\s+", " ", sql).strip()

    tables = []
    joins = []

    reserved = {
        "WHERE", "LEFT", "RIGHT", "INNER",
        "OUTER", "CROSS", "JOIN", "ON",
        "ORDER", "GROUP", "LIMIT", "HAVING",
        "OFFSET", "UNION", "FOR", "WINDOW"
    }

    from_match = re.search(
        r"\bFROM\s+([`\w.]+)(?:\s+(?:AS\s+)?(\w+))?",
        clean,
        re.I
    )

    if from_match:
        table = from_match.group(1).replace("`", "")
        alias = from_match.group(2)

        if alias and alias.upper() in reserved:
            alias = None

        tables.append({
            "table": table,
            "alias": alias or table.split(".")[-1]
        })

    join_pattern = re.compile(
        r"\b(INNER|LEFT|RIGHT|CROSS)?\s*"
        r"(?:OUTER\s+)?JOIN\s+([`\w.]+)"
        r"(?:\s+(?:AS\s+)?(\w+))?\s+ON\s+(.*?)"
        r"(?=\s+(?:(?:INNER|LEFT|RIGHT|CROSS)\s+)?"
        r"(?:OUTER\s+)?JOIN\b|"
        r"\s+WHERE\b|\s+GROUP\s+BY\b|"
        r"\s+HAVING\b|\s+ORDER\s+BY\b|"
        r"\s+LIMIT\b|\s+OFFSET\b|$)",
        re.I
    )

    for match in join_pattern.finditer(clean):
        join_type = (match.group(1) or "INNER").upper()
        table = match.group(2).replace("`", "")
        alias = match.group(3)

        if alias and alias.upper() in reserved:
            alias = None

        alias = alias or table.split(".")[-1]

        tables.append({
            "table": table,
            "alias": alias
        })

        joins.append({
            "type": join_type,
            "table": table,
            "alias": alias,
            "condition": match.group(4).strip()
        })

    def clause(pattern):
        match = re.search(pattern, clean, re.I)
        return match.group(1).strip() if match else None

    select = re.search(
        r"\bSELECT\s+(.*?)\s+FROM\b",
        clean,
        re.I
    )

    aggregates, windows = analyze_functions(clean)

    return {
        "tables": tables,
        "joins": joins,
        "where": clause(
            r"\bWHERE\s+(.*?)(?=\s+GROUP\s+BY\b|\s+HAVING\b|\s+ORDER\s+BY\b|\s+LIMIT\b|\s+OFFSET\b|$)"
        ),
        "group_by": clause(
            r"\bGROUP\s+BY\s+(.*?)(?=\s+HAVING\b|\s+ORDER\s+BY\b|\s+LIMIT\b|\s+OFFSET\b|$)"
        ),
        "having": clause(
            r"\bHAVING\s+(.*?)(?=\s+ORDER\s+BY\b|\s+LIMIT\b|\s+OFFSET\b|$)"
        ),
        "order_by": clause(
            r"\bORDER\s+BY\s+(.*?)(?=\s+LIMIT\b|\s+OFFSET\b|$)"
        ),
        "limit": clause(r"\bLIMIT\s+(\d+)"),
        "offset": clause(r"\bOFFSET\s+(\d+)"),
        "select": split_sql_expressions(
            select.group(1)
        ) if select else [],
        "aggregates": aggregates,
        "windows": windows
    }


# ============================================================
# IDENTIFICAR TABLAS FÍSICAS
# ============================================================

def find_table(name, tables):
    patterns = [
        r"Table scan on\s+`?([\w.]+)",
        r"Index scan on\s+`?([\w.]+)",
        r"Index lookup on\s+`?([\w.]+)",
        r"Index range scan on\s+`?([\w.]+)",
        r"Covering index lookup on\s+`?([\w.]+)",
        r"Covering index scan on\s+`?([\w.]+)"
    ]

    for pattern in patterns:
        match = re.search(pattern, name, re.I)

        if match:
            physical_name = match.group(1).replace("`", "")

            for item in tables:
                if (
                    item["alias"].lower() == physical_name.lower()
                    or item["table"].split(".")[-1].lower()
                    == physical_name.lower()
                ):
                    return item["table"]

            return physical_name

    for item in tables:
        alias = re.escape(item["alias"])

        if re.search(r"\b" + alias + r"\b", name, re.I):
            return item["table"]

    return None


def collect_tables(node, tables):
    found = []

    table = find_table(node["name"], tables)

    if table:
        found.append(table)

    for child in node["children"]:
        found.extend(collect_tables(child, tables))

    return list(dict.fromkeys(found))


# ============================================================
# DESCRIBIR AGREGACIONES
# ============================================================

def describe_aggregate_functions(sql_info):
    print("\nFUNCIONES DE AGREGACIÓN")

    if not sql_info["aggregates"]:
        print("   No se identificaron agregaciones.")
        return

    for item in sql_info["aggregates"]:
        function = item["function"]

        print("\n   FUNCIÓN:", function)
        print("   DESCRIPCIÓN:", AGGREGATE_FUNCTIONS[function])
        print("   ARGUMENTOS:", item["arguments"])

        columns = extract_columns(
            item["arguments"],
            sql_info["tables"]
        )

        for column in columns:
            print("   TABLA Y COLUMNA:", column)

        if item["arguments"] == "*":
            print("   CUENTA LAS FILAS DEL CONJUNTO DE ENTRADA.")

        print(
            "   TIPO:",
            "AGREGACIÓN VENTANA" if item["window"]
            else "AGREGACIÓN CONVENCIONAL"
        )


# ============================================================
# DESCRIBIR FUNCIONES VENTANA
# ============================================================

def describe_window_functions(sql_info):
    print("\nFUNCIONES VENTANA")

    if not sql_info["windows"]:
        print("   No se identificaron funciones ventana.")
        return

    for item in sql_info["windows"]:
        function = item["function"]

        print("\n   FUNCIÓN:", function)
        print("   DESCRIPCIÓN:", WINDOW_FUNCTIONS[function])
        print("   ARGUMENTOS:", item["arguments"])

        for column in extract_columns(
            item["arguments"],
            sql_info["tables"]
        ):
            print("   TABLA Y COLUMNA:", column)

        print(
            "   PARTITION BY:",
            item["partition_by"] or "Sin PARTITION BY"
        )

        print(
            "   ORDER BY:",
            item["order_by"] or "Sin ORDER BY"
        )

        print(
            "   FRAME:",
            item["frame"] or "Marco predeterminado de MySQL"
        )


# ============================================================
# DESCRIBIR OPERACIONES FÍSICAS
# ============================================================

def describe_physical_operation(node, sql_info):
    name = node["name"]
    lower_name = name.lower()

    operation, description = get_operation(name)

    print("\nDETALLE DE OPERACIÓN")
    print("   OPERACIÓN:", operation)
    print("   DESCRIPCIÓN:", description)

    table = find_table(name, sql_info["tables"])

    if "table scan" in lower_name:
        print("\n   TABLE SCAN")
        print("   NOMBRE FÍSICO DE TABLA:", table or "No identificada")
        print("   ACCESO: Lectura de registros de la tabla.")

    if "index" in lower_name:
        print("\n   OPERACIÓN DE ÍNDICE")
        print("   TABLA FÍSICA:", table or "No identificada")

        index = re.search(
            r"\busing\s+([^\s(]+)",
            name,
            re.I
        )

        print(
            "   ÍNDICE:",
            index.group(1) if index else "No identificado"
        )

    if "filter" in lower_name:
        print("\n   FILTER")
        print("   WHERE:", sql_info["where"] or "Sin WHERE")
        print("   HAVING:", sql_info["having"] or "Sin HAVING")

    if "sort" in lower_name:
        print("\n   SORT")
        print("   ORDER BY:", sql_info["order_by"] or "Ordenamiento interno")

    if "aggregate" in lower_name:
        print("\n   AGGREGATE")
        print("   GROUP BY:", sql_info["group_by"] or "Sin GROUP BY explícito")
        print("   HAVING:", sql_info["having"] or "Sin HAVING")
        describe_aggregate_functions(sql_info)

    if "window" in lower_name:
        describe_window_functions(sql_info)

    if "limit" in lower_name:
        print("\n   LIMIT")
        print("   CANTIDAD:", sql_info["limit"] or "No identificada")
        print("   OFFSET:", sql_info["offset"] or "0")

    if "offset" in lower_name:
        print("\n   OFFSET")
        print("   FILAS OMITIDAS:", sql_info["offset"] or "No identificadas")


# ============================================================
# IDENTIFICAR COLUMNAS DE UNIÓN
# ============================================================

def extract_join_relationship(condition, tables):
    relationships = []

    pattern = re.compile(
        r"([\w`]+\.[\w`]+)\s*(=|<>|!=|>=|<=|>|<)\s*([\w`]+\.[\w`]+)",
        re.I
    )

    for match in pattern.finditer(condition):
        left = resolve_column_reference(
            match.group(1),
            tables
        )

        right = resolve_column_reference(
            match.group(3),
            tables
        )

        relationships.append({
            "left": left,
            "operator": match.group(2),
            "right": right
        })

    return relationships


def normalize_join_condition(condition, tables):
    result = condition

    for item in sorted(
        tables,
        key=lambda x: len(x["alias"]),
        reverse=True
    ):
        alias = re.escape(item["alias"])

        result = re.sub(
            r"(?<![\w.])`?" + alias + r"`?(?=\s*\.)",
            item["table"],
            result,
            flags=re.I
        )

    return result


# ============================================================
# IDENTIFICAR JOIN ORIGINAL
# ============================================================

def match_physical_join(left_tables, right_tables, sql_info):
    candidates = []

    for join in sql_info["joins"]:
        relationships = extract_join_relationship(
            join["condition"],
            sql_info["tables"]
        )

        score = 0

        for relation in relationships:
            left_table = relation["left"].rsplit(".", 1)[0]
            right_table = relation["right"].rsplit(".", 1)[0]

            if (
                left_table in left_tables
                and right_table in right_tables
            ) or (
                right_table in left_tables
                and left_table in right_tables
            ):
                score += 10

        if score:
            candidates.append((score, join))

    if not candidates:
        return None

    candidates.sort(
        key=lambda item: item[0],
        reverse=True
    )

    if (
        len(candidates) > 1
        and candidates[0][0] == candidates[1][0]
    ):
        return None

    return candidates[0][1]


# ============================================================
# DESCRIBIR JOIN COMPLETO
# ============================================================

def describe_join(node, sql_info):
    name = node["name"].lower()

    if "join" not in name:
        return

    print("\n" + "=" * 90)
    print("DETALLE DEL JOIN FÍSICO")
    print("=" * 90)

    left_tables = (
        collect_tables(
            node["children"][0],
            sql_info["tables"]
        )
        if len(node["children"]) >= 1
        else []
    )

    right_tables = (
        collect_tables(
            node["children"][1],
            sql_info["tables"]
        )
        if len(node["children"]) >= 2
        else []
    )

    print("\n1. ENTRADA IZQUIERDA")

    for table in left_tables:
        print("   TABLA FÍSICA:", table)

    if not left_tables:
        print("   Resultado intermedio o no identificado.")

    print("\n2. ENTRADA DERECHA")

    for table in right_tables:
        print("   TABLA FÍSICA:", table)

    if not right_tables:
        print("   Resultado intermedio o no identificado.")

    print("\n3. OPERACIÓN FÍSICA")

    if "antijoin" in name:
        print("   NESTED LOOP ANTIJOIN")
    elif "nested loop left join" in name:
        print("   NESTED LOOP LEFT JOIN")
    elif "nested loop inner join" in name:
        print("   NESTED LOOP INNER JOIN")
    elif "hash join" in name:
        print("   HASH JOIN")
    else:
        print("  ", node["name"])

    matching = match_physical_join(
        left_tables,
        right_tables,
        sql_info
    )

    print("\n4. JOIN ORIGINAL DEL SQL")

    if matching:
        print("   TIPO:", matching["type"], "JOIN")
        print("   TABLA FÍSICA INCORPORADA:", matching["table"])

        print("\n5. CONDICIÓN ON")

        print(
            "   ON",
            normalize_join_condition(
                matching["condition"],
                sql_info["tables"]
            )
        )

        print("\n6. COLUMNAS DE UNIÓN")

        relationships = extract_join_relationship(
            matching["condition"],
            sql_info["tables"]
        )

        for relation in relationships:
            print("\n   TABLA/COLUMNA IZQUIERDA:", relation["left"])
            print("   OPERADOR:", relation["operator"])
            print("   TABLA/COLUMNA DERECHA:", relation["right"])

    else:
        print(
            "   No se pudo identificar inequívocamente "
            "el JOIN original correspondiente."
        )

    print("\n7. RESUMEN DE ENTRADAS")
    print("   IZQUIERDA:", ", ".join(left_tables) or "No identificada")
    print("   DERECHA:", ", ".join(right_tables) or "No identificada")

    print("=" * 90)


# ============================================================
# GRAPHVIZ: COLORES DIFERENTES Y ÁRBOL DE ABAJO HACIA ARRIBA
# ============================================================

def generate_graphviz(root, sql_info):
    graph = Digraph(
        "Execution Plan",
        format="png"
    )

    graph.attr(
        rankdir="BT",
        ordering="out",
        bgcolor="white",
        nodesep="0.5",
        ranksep="0.8",
        splines="ortho"
    )

    graph.attr(
        "node",
        shape="box",
        style="filled,rounded",
        fontname="Arial",
        fontsize="10",
        color="#333333",
        penwidth="1.5",
        margin="0.15"
    )

    graph.attr(
        "edge",
        color="#555555",
        arrowsize="0.8"
    )

    COLORS = {
        "LIMIT": "#FFCC80",
        "TABLE SCAN": "#90CAF9",
        "INDEX SCAN": "#80DEEA",
        "INDEX RANGE SCAN": "#80CBC4",
        "INDEX LOOKUP": "#A5D6A7",
        "COVERING INDEX LOOKUP": "#C5E1A5",
        "COVERING INDEX SCAN": "#AED581",
        "SINGLE ROW INDEX LOOKUP": "#B2DFDB",
        "NESTED LOOP INNER JOIN": "#CE93D8",
        "NESTED LOOP LEFT JOIN": "#F48FB1",
        "NESTED LOOP ANTIJOIN": "#EF9A9A",
        "HASH JOIN": "#B39DDB",
        "FILTER": "#FFF59D",
        "SORT": "#FFAB91",
        "AGGREGATE": "#80CBC4",
        "GROUP AGGREGATE": "#4DB6AC",
        "MATERIALIZE": "#B0BEC5",
        "REMOVE DUPLICATES": "#F8BBD0",
        "STREAM RESULTS": "#C5CAE9",
        "WINDOW": "#7986CB",
        "WINDOW AGGREGATE": "#9FA8DA",
        "OFFSET": "#FFE082"
    }

    counter = [0]

    def add_nodes(node, parent_id=None):
        counter[0] += 1
        node_id = f"node_{counter[0]}"

        operation, description = get_operation(
            node["name"]
        )

        color = COLORS.get(operation, "#E0E0E0")

        metrics = node["metrics"]

        label = (
            f"{operation}\n"
            f"{description}\n"
            f"Operación física: {node['name']}"
        )

        table = find_table(
            node["name"],
            sql_info["tables"]
        )

        if table:
            label += f"\nTabla física: {table}"

        index = re.search(
            r"\busing\s+([^\s(]+)",
            node["name"],
            re.I
        )

        if index:
            label += f"\nÍndice: {index.group(1)}"

        if "join" in node["name"].lower():
            left_tables = (
                collect_tables(
                    node["children"][0],
                    sql_info["tables"]
                )
                if len(node["children"]) >= 1
                else []
            )

            right_tables = (
                collect_tables(
                    node["children"][1],
                    sql_info["tables"]
                )
                if len(node["children"]) >= 2
                else []
            )

            label += (
                "\n\nENTRADA IZQUIERDA:\n"
                + (", ".join(left_tables) or "Resultado intermedio")
            )

            label += (
                "\nENTRADA DERECHA:\n"
                + (", ".join(right_tables) or "Resultado intermedio")
            )

            matching = match_physical_join(
                left_tables,
                right_tables,
                sql_info
            )

            if matching:
                label += (
                    "\nJOIN SQL: "
                    + matching["type"]
                    + " JOIN"
                )

                label += (
                    "\nON: "
                    + normalize_join_condition(
                        matching["condition"],
                        sql_info["tables"]
                    )
                )

        if metrics["cost"]:
            label += f"\nCoste estimado: {metrics['cost']}"

        if metrics["estimated"]:
            label += f"\nFilas estimadas: {metrics['estimated']}"

        if metrics["actual"]:
            label += f"\nFilas reales: {metrics['actual']}"

        if metrics["start"] and metrics["end"]:
            label += (
                f"\nTiempo: {metrics['start']}.."
                f"{metrics['end']} ms"
            )

        if metrics["loops"]:
            label += f"\nLoops: {metrics['loops']}"

        graph.node(
            node_id,
            label=label,
            fillcolor=color
        )

        if parent_id:
            graph.edge(parent_id, node_id)

        for child in node["children"]:
            add_nodes(child, node_id)

    add_nodes(root)

    output_path = graph.render(
        filename="execution_plan",
        directory=".",
        cleanup=True,
        view=False
    )

    print("\nGRAPHVIZ GENERADO:")
    print(os.path.abspath(output_path))

    return output_path


# ============================================================
# ANALIZAR CLÁUSULAS SQL
# ============================================================

def explain_sql_operations(sql_info):
    print("\n" + "=" * 100)
    print("ANÁLISIS DE CLÁUSULAS SQL")
    print("=" * 100)

    print("\n1. GROUP BY")
    print("   ", sql_info["group_by"] or "Sin GROUP BY")

    if sql_info["group_by"]:
        for expression in split_sql_expressions(
            sql_info["group_by"]
        ):
            print(
                "   TABLA/COLUMNA:",
                resolve_column_reference(
                    expression,
                    sql_info["tables"]
                )
            )

    print("\n2. HAVING")
    print("   ", sql_info["having"] or "Sin HAVING")

    print("\n3. FUNCIONES DE AGREGACIÓN")
    describe_aggregate_functions(sql_info)

    print("\n4. FUNCIONES VENTANA")
    describe_window_functions(sql_info)

    print("\n5. ORDER BY")
    print("   ", sql_info["order_by"] or "Sin ORDER BY")

    print("\n6. LIMIT")
    print("   ", sql_info["limit"] or "Sin LIMIT")

    print("\n7. OFFSET")
    print("   ", sql_info["offset"] or "Sin OFFSET")


# ============================================================
# EXPLICACIÓN DEL ÁRBOL
# ============================================================

def explain_tree(explain_result, sql_query):
    sql_info = analyze_sql(sql_query)

    for row in explain_result:
        plan_text = row[0] if isinstance(row, (tuple, list)) else str(row)

        root = build_tree(plan_text)

        if not root:
            continue

        print("\n" + "=" * 100)
        print("ANÁLISIS DEL PLAN DE EJECUCIÓN MYSQL")
        print("=" * 100)

        print("\n1. TABLAS INVOLUCRADAS")

        for table in sql_info["tables"]:
            print(
                "   TABLA FÍSICA:",
                table["table"]
            )

        print("\n2. JOINS DECLARADOS")

        for i, join in enumerate(sql_info["joins"], 1):
            print(f"\n   JOIN #{i}")
            print("   Tipo:", join["type"], "JOIN")
            print("   Tabla física:", join["table"])

            print(
                "   Condición ON:",
                normalize_join_condition(
                    join["condition"],
                    sql_info["tables"]
                )
            )

        print("\n3. CONDICIÓN WHERE")
        print("   ", sql_info["where"] or "Sin WHERE")

        explain_sql_operations(sql_info)

        sequence = get_execution_sequence(root)

        print("\n" + "=" * 100)
        print("OPERACIONES EN ORDEN DE EJECUCIÓN")
        print("=" * 100)

        for i, node in enumerate(sequence, 1):
            operation, description = get_operation(node["name"])
            metrics = node["metrics"]

            print(f"\n{'─' * 90}")
            print(f"PASO {i}: {operation}")
            print(f"{'─' * 90}")

            print("Operación física:", node["name"])
            print("Descripción:", description)

            describe_physical_operation(
                node,
                sql_info
            )

            if "join" in node["name"].lower():
                describe_join(
                    node,
                    sql_info
                )

            if metrics["cost"]:
                print("\nMÉTRICAS")
                print("Coste estimado:", metrics["cost"])
                print("Filas estimadas:", metrics["estimated"])

            if metrics["actual"]:
                actual = int(metrics["actual"])
                loops = int(metrics["loops"] or 1)

                print("Filas reales por ejecución:", actual)
                print("Tiempo inicial:", metrics["start"], "ms")
                print("Tiempo final:", metrics["end"], "ms")
                print("Loops:", loops)
                print("Filas acumuladas aproximadas:", actual * loops)

        print("\n" + "=" * 100)
        print("ÁRBOL JERÁRQUICO")
        print("=" * 100)

        def render(node, prefix="", last=True, root_node=False):
            operation, _ = get_operation(node["name"])

            connector = "" if root_node else (
                "└── " if last else "├── "
            )

            print(prefix + connector + operation)

            new_prefix = prefix + (
                "    " if last or root_node else "│   "
            )

            for i, child in enumerate(node["children"]):
                render(
                    child,
                    new_prefix,
                    i == len(node["children"]) - 1
                )

        render(root, root_node=True)

        # GENERAR IMAGEN GRAPHVIZ
        generate_graphviz(
            root,
            sql_info
        )

        print("\n" + "=" * 100)


# ============================================================
# CONSTRUIR EL ÁRBOL DEL PLAN DE EJECUCIÓN MYSQL
# ============================================================

def build_tree(plan_text):

    root = None
    stack = []

    for line in plan_text.splitlines():

        if not line.strip():
            continue

        # Identificar profundidad según indentación
        indent = len(line) - len(line.lstrip())
        depth = indent // 4

        name, metrics = parse_line(line)

        node = {
            "name": name,
            "depth": depth,
            "metrics": metrics,
            "children": []
        }

        # Buscar el padre correspondiente
        while stack and stack[-1]["depth"] >= depth:
            stack.pop()

        if stack:
            stack[-1]["children"].append(node)
        else:
            root = node

        stack.append(node)

    return root


# ============================================================
# OBTENER OPERACIONES EN ORDEN DE EJECUCIÓN
# ============================================================

def get_execution_sequence(node, sequence=None):

    if sequence is None:
        sequence = []

    # Primero se ejecutan las operaciones hijas
    for child in node["children"]:
        get_execution_sequence(child, sequence)

    # Después se procesa el nodo padre
    sequence.append(node)

    return sequence


# ============================================================
# EJECUTAR CONSULTAS MYSQL Y GENERAR GRAPHVIZ
# ============================================================

def execute_query(
    query,
    params=None,
    show_results=False,
    explain_analyse=False,
    show_query=False
):

    connection = None
    cursor = None

    try:

        connection = mysql.connector.connect(**MYSQL_CONFIG)

        cursor = connection.cursor()

        query = re.sub(
            r"^\s*EXPLAIN\s+ANALYZE\s+",
            "",
            query,
            flags=re.I
        ).strip().rstrip(";")

        if explain_analyse:

            final_query = "EXPLAIN ANALYZE " + query

        else:

            final_query = query

        if show_query:

            print("\n" + "=" * 90)
            print("CONSULTA EJECUTADA")
            print("=" * 90)

            print(final_query)

        cursor.execute(
            final_query,
            params or ()
        )

        if cursor.with_rows:

            results = cursor.fetchall()

            if explain_analyse:

                print("\nPLAN DE EJECUCIÓN MYSQL")

                explain_tree(
                    results,
                    sql_query=query
                )

            elif show_results:

                print("\nRESULTADOS DE LA CONSULTA")

                for row in results:
                    print(row)

            return results

        connection.commit()

        return cursor.rowcount

    except mysql.connector.Error as e:

        if connection and connection.is_connected():
            connection.rollback()

        print("\nERROR MYSQL:", e)

        return None

    except Exception as e:

        print("\nERROR DE EJECUCIÓN:", e)

        return None

    finally:

        if cursor:
            cursor.close()

        if connection and connection.is_connected():
            connection.close()

# ============================================================
# EJEMPLO DE USO
# ============================================================

query = """

SELECT
    c.category_id,
    c.name AS category_name,

    COUNT(p.product_id) AS total_products,
    SUM(p.stock) AS total_stock,
    ROUND(AVG(p.price), 2) AS average_price,
    MIN(p.price) AS minimum_price,
    MAX(p.price) AS maximum_price,

    ROW_NUMBER() OVER (
        ORDER BY COUNT(p.product_id) DESC
    ) AS numero_fila,

    RANK() OVER (
        ORDER BY AVG(p.price) DESC
    ) AS ranking_precio,

    DENSE_RANK() OVER (
        ORDER BY SUM(p.stock) DESC
    ) AS ranking_stock,

    LAG(AVG(p.price)) OVER (
        ORDER BY c.category_id
    ) AS precio_categoria_anterior,

    LEAD(AVG(p.price)) OVER (
        ORDER BY c.category_id
    ) AS precio_categoria_siguiente,

    SUM(COUNT(p.product_id)) OVER () AS total_general

FROM categories c

INNER JOIN products p
    ON c.category_id = p.category_id

GROUP BY
    c.category_id,
    c.name

HAVING
    COUNT(p.product_id) >= 1
    AND SUM(p.stock) > 0
    AND AVG(p.price) > 0

ORDER BY
    total_products DESC,
    average_price DESC

LIMIT 100
OFFSET 1;
"""

result = execute_query(
    query=query,
    explain_analyse=True,
    show_query=True
)

/usr/bin/xdg-open: 1032: www-browser: not found
/usr/bin/xdg-open: 1032: links2: not found
/usr/bin/xdg-open: 1032: elinks: not found
/usr/bin/xdg-open: 1032: links: not found
/usr/bin/xdg-open: 1032: lynx: not found
/usr/bin/xdg-open: 1032: w3m: not found
xdg-open: no method available for opening 'execution_plan.png'



CONSULTA EJECUTADA
EXPLAIN ANALYZE SELECT
    c.category_id,
    c.name AS category_name,

    COUNT(p.product_id) AS total_products,
    SUM(p.stock) AS total_stock,
    ROUND(AVG(p.price), 2) AS average_price,
    MIN(p.price) AS minimum_price,
    MAX(p.price) AS maximum_price,

    ROW_NUMBER() OVER (
        ORDER BY COUNT(p.product_id) DESC
    ) AS numero_fila,

    RANK() OVER (
        ORDER BY AVG(p.price) DESC
    ) AS ranking_precio,

    DENSE_RANK() OVER (
        ORDER BY SUM(p.stock) DESC
    ) AS ranking_stock,

    LAG(AVG(p.price)) OVER (
        ORDER BY c.category_id
    ) AS precio_categoria_anterior,

    LEAD(AVG(p.price)) OVER (
        ORDER BY c.category_id
    ) AS precio_categoria_siguiente,

    SUM(COUNT(p.product_id)) OVER () AS total_general

FROM categories c

INNER JOIN products p
    ON c.category_id = p.category_id

GROUP BY
    c.category_id,
    c.name

HAVING
    COUNT(p.product_id) >= 1
    AND SUM(p.stock) > 0
    AND AVG(p.price) > 0

ORDER B

In [3]:
import pandas as pd
from sqlalchemy import create_engine, text

def get_pandas_connection(config):

    connection_string = (
        f"mysql+mysqlconnector://"
        f"{config['user']}:"
        f"{config['password']}@"
        f"{config['host']}:"
        f"{config['port']}/"
        f"{config['database']}"
    )

    return create_engine(connection_string)


# ============================================================
# EJECUTAR QUERY MYSQL CON PANDAS
# ============================================================

def execute_query_pandas(
    query,
    params=None,
    show_results=False,
    config=None
):

    engine = None

    try:

        engine = get_pandas_connection(config)

        df = pd.read_sql_query(
            text(query),
            engine,
            params=params
        )

        if show_results:

            print("\nRESULTADOS DE PANDAS")
            print("=" * 90)

            print(df.to_string(index=False))

            print("\nDIMENSIONES DEL DATAFRAME")
            print("Filas:", df.shape[0])
            print("Columnas:", df.shape[1])

            print("\nTIPOS DE DATOS")
            print(df.dtypes)

        return df

    except Exception as e:

        print("ERROR PANDAS/MYSQL:", e)

        return None

    finally:

        if engine:
            engine.dispose()


# ============================================================
# EJEMPLO DE USO
# ============================================================
df = execute_query_pandas(
    query=query,
    show_results=False,
    config=MYSQL_CONFIG
)

df

,category_id,category_name,total_products,total_stock,average_price,minimum_price,maximum_price,numero_fila,ranking_precio,ranking_stock,precio_categoria_anterior,precio_categoria_siguiente,total_general
0,2,Software,20,19980.0,169.71,19.99,899.00,1,5,1,155.8625,186.6150,330.0
1,1,Electronics,20,1473.0,155.86,19.99,499.99,15,6,14,NaN,169.7140,330.0
2,15,Music & Instruments,20,2243.0,55.94,5.99,189.99,2,7,7,43.1670,3515.0000,330.0
3,8,Home & Kitchen,20,1835.0,45.46,9.99,189.99,9,8,12,14.2645,23.8420,330.0
4,14,Outdoor & Garden,20,1650.0,43.17,12.99,149.00,3,9,13,36.4390,55.9415,330.0
5,13,Clothing & Apparel,20,2065.0,36.44,12.99,89.99,4,10,11,24.5410,43.1670,330.0
6,5,Sports,20,2195.0,31.74,9.99,79.99,12,11,8,31.0670,27.1890,330.0
7,4,Accessories,20,2570.0,31.07,9.99,89.99,13,12,3,186.6150,31.7390,330.0
8,10,Automotive,20,2365.0,27.79,9.99,69.99,7,13,5,23.8420,23.5665,330.0
9,6,Toys & Games,20,2070.0,27.19,8.99,89.99,11,14,10,31.7390,14.2645,330.0
